# Session 3: Overfitting — Seeing It Happen

## The Story

Your team proudly presents a churn model: **97% accurate!**

Before you celebrate, let's see what happens when we push a model too far — and how to catch it.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

In [ ]:
# Create a dataset with 8 real features and some noise features
n = 3000

data = pd.DataFrame({
    # Real predictors
    'tenure_years': np.random.exponential(4, n).clip(0, 20).astype(int),
    'premium_increase': np.random.normal(5, 8, n).round(1),
    'claims_3y': np.random.poisson(0.5, n),
    'age': np.random.normal(45, 13, n).clip(18, 80).astype(int),
    'multi_policy': np.random.choice([0, 1], n, p=[0.6, 0.4]),
    # Noise features (no real predictive power)
    'day_of_birth': np.random.randint(1, 29, n),
    'name_length': np.random.randint(4, 15, n),
    'policy_number_last_digit': np.random.randint(0, 10, n),
    'agent_desk_number': np.random.randint(1, 50, n),
    'zodiac_numeric': np.random.randint(1, 13, n),
})

# Target depends ONLY on real features
risk = (
    0.3 * (data['premium_increase'] > 10).astype(float) +
    0.25 * (data['tenure_years'] < 2).astype(float) +
    0.2 * (data['claims_3y'] == 0).astype(float) +
    0.15 * (data['multi_policy'] == 0).astype(float) +
    0.1 * (data['age'] < 30).astype(float) +
    np.random.normal(0, 0.25, n)
)
data['lapsed'] = (risk > np.percentile(risk, 85)).astype(int)

features = data.columns[:-1].tolist()
X = data[features]
y = data['lapsed']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print("📋 SETUP")
print("=" * 50)
print(f"  Training set: {len(X_train):,} customers")
print(f"  Test set: {len(X_test):,} customers (unseen)")
print(f"  Lapse rate: {y.mean():.1%}")
print(f"\n  Features (10 total):")
print(f"    Real predictors: tenure, premium_increase, claims, age, multi_policy")
print(f"    Noise features:  day_of_birth, name_length, policy_number, desk_number, zodiac")
print(f"\n  ⚠️ The model doesn't know which features are real and which are noise!")

## The Complexity Experiment

Let's build decision trees of increasing depth and watch what happens to training vs. test performance:

In [ ]:
# Build trees of increasing complexity
depths = range(1, 21)
train_scores = []
test_scores = []

for d in depths:
    tree = DecisionTreeClassifier(max_depth=d, random_state=42)
    tree.fit(X_train, y_train)
    train_scores.append(roc_auc_score(y_train, tree.predict_proba(X_train)[:, 1]))
    test_scores.append(roc_auc_score(y_test, tree.predict_proba(X_test)[:, 1]))

# Find the sweet spot
best_depth = depths[np.argmax(test_scores)]
best_test = max(test_scores)

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.plot(depths, train_scores, 'o-', color='#00008f', lw=2.5, label='Training performance')
ax.plot(depths, test_scores, 's-', color='#FF1721', lw=2.5, label='Test performance (unseen data)')
ax.axvline(x=best_depth, color='green', linestyle='--', alpha=0.7, lw=2, label=f'Sweet spot (depth={best_depth})')

# Annotate regions
ax.axvspan(0.5, 3.5, alpha=0.05, color='blue', label='_')
ax.axvspan(best_depth + 2, 20.5, alpha=0.05, color='red', label='_')
ax.text(2, min(train_scores) + 0.02, 'Underfitting\n(too simple)', ha='center', fontsize=10, color='blue')
ax.text(16, min(train_scores) + 0.02, 'Overfitting\n(too complex)', ha='center', fontsize=10, color='red')

ax.set_xlabel('Tree Depth (complexity)', fontsize=12)
ax.set_ylabel('AUC Score', fontsize=12)
ax.set_title('The Overfitting Curve: Training vs. Test Performance', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n📊 KEY OBSERVATIONS:")
print(f"  • Training performance always IMPROVES with more complexity")
print(f"  • Test performance improves, peaks at depth {best_depth}, then DEGRADES")
print(f"  • The GAP between lines = overfitting")
print(f"\n  At depth 20: Train AUC = {train_scores[-1]:.3f}, Test AUC = {test_scores[-1]:.3f}")
print(f"  At depth {best_depth}:  Train AUC = {train_scores[best_depth-1]:.3f}, Test AUC = {best_test:.3f}")
print(f"\n  💡 The deep tree MEMORIZED the training data but fails on new customers.")

### 🤔 Discussion

- If someone only showed you the TRAINING score, would you approve deployment?
- What does the widening gap tell you about trust in the model?

---

## Cross-Validation: Don't Rely on One Test

A single train/test split might be lucky or unlucky. Cross-validation runs 5 independent tests:

In [ ]:
# Cross-validation for the sweet-spot tree vs. the overfit tree
tree_simple = DecisionTreeClassifier(max_depth=best_depth, random_state=42)
tree_overfit = DecisionTreeClassifier(max_depth=20, random_state=42)

cv_simple = cross_val_score(tree_simple, X, y, cv=5, scoring='roc_auc')
cv_overfit = cross_val_score(tree_overfit, X, y, cv=5, scoring='roc_auc')

print("📊 CROSS-VALIDATION: 5 Independent Tests")
print("=" * 55)
print(f"\n  Balanced tree (depth {best_depth}):")
print(f"    Scores: {[f'{s:.3f}' for s in cv_simple]}")
print(f"    Mean: {cv_simple.mean():.3f} ± {cv_simple.std():.3f}")

print(f"\n  Overfit tree (depth 20):")
print(f"    Scores: {[f'{s:.3f}' for s in cv_overfit]}")
print(f"    Mean: {cv_overfit.mean():.3f} ± {cv_overfit.std():.3f}")

print(f"\n  💡 The balanced tree is:")
print(f"     {'Better' if cv_simple.mean() > cv_overfit.mean() else 'Similar'} on average ({cv_simple.mean():.3f} vs {cv_overfit.mean():.3f})")
print(f"     More STABLE (lower variance: ±{cv_simple.std():.3f} vs ±{cv_overfit.std():.3f})")

## What the Overfit Tree Learned (That It Shouldn't Have)

In [ ]:
# Show what the overfit tree thinks matters
tree_overfit.fit(X_train, y_train)
tree_simple.fit(X_train, y_train)

imp_overfit = pd.Series(tree_overfit.feature_importances_, index=features).sort_values(ascending=True)
imp_simple = pd.Series(tree_simple.feature_importances_, index=features).sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

noise_features = ['day_of_birth', 'name_length', 'policy_number_last_digit', 'agent_desk_number', 'zodiac_numeric']
colors_simple = ['#FF1721' if f in noise_features else '#00008f' for f in imp_simple.index]
colors_overfit = ['#FF1721' if f in noise_features else '#00008f' for f in imp_overfit.index]

imp_simple.plot(kind='barh', ax=axes[0], color=colors_simple, alpha=0.8)
axes[0].set_title(f'Balanced Tree (depth {best_depth})', fontweight='bold')
axes[0].set_xlabel('Importance')

imp_overfit.plot(kind='barh', ax=axes[1], color=colors_overfit, alpha=0.8)
axes[1].set_title('Overfit Tree (depth 20)', fontweight='bold')
axes[1].set_xlabel('Importance')

import matplotlib.patches as mpatches
legend_elements = [
    mpatches.Patch(facecolor='#00008f', alpha=0.8, label='Real features'),
    mpatches.Patch(facecolor='#FF1721', alpha=0.8, label='Noise features'),
]
fig.legend(handles=legend_elements, loc='upper center', ncol=2, fontsize=11, bbox_to_anchor=(0.5, 1.02))

plt.suptitle('Overfitting: The Complex Tree Trusts Noise', fontsize=13, fontweight='bold', y=1.06)
plt.tight_layout()
plt.show()

print("\n→ Left: The balanced tree correctly ignores noise features (red bars are tiny).")
print("  Right: The overfit tree assigns REAL importance to noise like 'zodiac sign' and 'desk number'.")
print("\n  ⚠️ The overfit model 'learned' that customers born on the 14th are riskier.")
print("     That's obviously nonsense — but the model found a spurious pattern in training data.")

---

## The Out-of-Time Test: The Ultimate Check

In insurance, the strongest validation isn't random split — it's **out-of-time testing.**

Train on 2020-2022 data. Test on 2023 data. If it still works → it generalizes.

In [ ]:
# Simulate an out-of-time scenario
# "2023" has slightly different distributions (post-inflation shift)
n_future = 1000

future_data = pd.DataFrame({
    'tenure_years': np.random.exponential(4, n_future).clip(0, 20).astype(int),
    'premium_increase': np.random.normal(8, 9, n_future).round(1),  # Higher increases in 2023!
    'claims_3y': np.random.poisson(0.5, n_future),
    'age': np.random.normal(45, 13, n_future).clip(18, 80).astype(int),
    'multi_policy': np.random.choice([0, 1], n_future, p=[0.55, 0.45]),
    'day_of_birth': np.random.randint(1, 29, n_future),
    'name_length': np.random.randint(4, 15, n_future),
    'policy_number_last_digit': np.random.randint(0, 10, n_future),
    'agent_desk_number': np.random.randint(1, 50, n_future),
    'zodiac_numeric': np.random.randint(1, 13, n_future),
})

future_risk = (
    0.3 * (future_data['premium_increase'] > 10).astype(float) +
    0.25 * (future_data['tenure_years'] < 2).astype(float) +
    0.2 * (future_data['claims_3y'] == 0).astype(float) +
    0.15 * (future_data['multi_policy'] == 0).astype(float) +
    0.1 * (future_data['age'] < 30).astype(float) +
    np.random.normal(0, 0.25, n_future)
)
future_data['lapsed'] = (future_risk > np.percentile(future_risk, 85)).astype(int)

X_future = future_data[features]
y_future = future_data['lapsed']

auc_simple_future = roc_auc_score(y_future, tree_simple.predict_proba(X_future)[:, 1])
auc_overfit_future = roc_auc_score(y_future, tree_overfit.predict_proba(X_future)[:, 1])

print("📊 OUT-OF-TIME TEST: Models Trained on 2020-2022, Tested on 2023")
print("=" * 60)
print(f"  (2023 has higher premium increases due to inflation)")
print(f"\n  {'Model':<25} {'Random Split':<18} {'Out-of-Time 2023':<18} {'Drop'}")
print("  " + "-" * 60)
print(f"  {'Balanced (depth '+str(best_depth)+')':<25} {best_test:<18.3f} {auc_simple_future:<18.3f} {best_test - auc_simple_future:+.3f}")
print(f"  {'Overfit (depth 20)':<25} {test_scores[-1]:<18.3f} {auc_overfit_future:<18.3f} {test_scores[-1] - auc_overfit_future:+.3f}")
print(f"\n  💡 The overfit model degrades MORE when the world changes.")
print(f"     The balanced model is more ROBUST to distribution shifts.")

---

## Key Takeaways

1. **Training accuracy is ALWAYS optimistic** — never deploy based on training scores alone
2. **The gap between training and test performance = overfitting** — monitor this gap
3. **Complex models find spurious patterns** — they'll happily use zodiac signs if you let them
4. **Cross-validation gives robust estimates** — multiple tests > one lucky split
5. **Out-of-time testing is the gold standard** — especially when the world is changing

> The question to ask: *"What's the performance gap between training and test? And did you test on future-like data?"*